In [1]:
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. 初期パラメータ設定 (Experiment Configurations)
# =====================================================================
# [モデル設定]
N = 40                  # 変数の数 (観測の数 P も今回は同じ40)
F = 8.0                 # 強制項
dt = 0.01               # 積分時間ステップ
sampling_dt = 0.05      # 観測・同化の間隔 (6時間相当: 0.25日/5日)
m = 8                   # アンサンブルメンバー数
p = 40                  # 観測点の数 (今回は全点観測なので N と同じ)

# [時間設定]
years = 2
units_per_year = 73.0   # 1年 = 365日 / 5日 = 73ユニット
total_time = years * units_per_year
spin_up_time = units_per_year

steps_total = int(total_time / dt)
steps_spin_up = int(spin_up_time / dt)
sampling_interval = int(sampling_dt / dt)

# [データ同化設定]
H_mat = np.eye(N)                       # 観測演算子 H (全点観測)
R_mat = np.eye(N)                       # 観測誤差共分散 R (対角成分1)

# =====================================================================
# 2. Nature Run (真値) と Observation (観測) の生成
# =====================================================================
def lorenz96(x, F):
    return (np.roll(x, -1, axis=0) - np.roll(x, 2, axis=0)) * np.roll(x, 1, axis=0) - x + F

def M(x_in, dt, steps):
    x_out = x_in.copy()
    for _ in range(steps):
        k1 = lorenz96(x_out, F)
        k2 = lorenz96(x_out + k1 * (dt / 2.0), F)
        k3 = lorenz96(x_out + k2 * (dt / 2.0), F)
        k4 = lorenz96(x_out + k3 * dt, F)
        x_out += (k1 + 2.0*k2 + 2.0*k3 + k4) * (dt / 6.0)
    return x_out

print("Nature Runと観測データを生成中...")
x = np.full(N, F)
x[19] += 1.001

true_states = []
for s in range(steps_total):
    x = M(x, dt, 1)
    if s >= steps_spin_up and (s - steps_spin_up) % sampling_interval == 0:
        true_states.append(x.copy())
true_states = np.array(true_states)

rng_obs = np.random.default_rng(seed=67) 
noise = rng_obs.normal(loc=0.0, scale=1.0, size=true_states.shape)
noise -= np.mean(noise, axis=0)
y_o_data_full = true_states + noise

num_cycles = y_o_data_full.shape[0]

print("生成完了！")

Nature Runと観測データを生成中...
生成完了！


In [ ]:
def get_experiment_matrices(obs_indices):
    p_obs = len(obs_indices)
    H_mat = np.zeros((p_obs, N))
    for k, j in enumerate(obs_indices):
        H_mat[k, j] = 1.0
    R_mat = np.eye(p_obs)
    return H_mat, R_mat

def R_localization_inv(sigma):
    R_loc_inv = np.zeros((N, N))
    def L(d):
        if d < np.sqrt(10 / 3) * sigma * 2.0:
            return np.exp(- (d**2) / (2 * sigma**2))
        else:
            return 0.0
    for i in range(N):
        for j in range(N):
            d = min(abs(i - j), N - abs(i - j))
            orig_r_inv = 1.0 / R_mat[j, j]
            R_loc_inv[i, j] = orig_r_inv * L(d)
    return R_loc_inv

def LETKF_core( 
    y_o_data,  
    true_states,
    sigma,  
    W_weights_list, 
):
    num_cycles = y_o_data.shape[0]
    R_loc_inv = R_localization_inv(sigma)
    
    record_rmse = np.zeros(num_cycles)
    record_spread = np.zeros(num_cycles)
    
    # 初期アンサンブルの生成
    rng_enkf = np.random.default_rng(seed=42)
    init_noise = rng_enkf.normal(0.0, 0.1, size=(N, m))
    init_noise -= np.mean(init_noise, axis=1, keepdims=True)
    
    x_raw_init = np.full(N, F)
    x_raw_init[min(19, N-1)] += 1.001
    X_a = M(x_raw_init[:, None] + init_noise, dt, steps_spin_up)
    
    delta = 1
    
    for t in range(num_cycles):
        y_o = y_o_data[t]
        
        X_b = M(X_a, dt, sampling_interval)
        X_b_mean = np.mean(X_b, axis=1, keepdims=True)
        dX_b = X_b - X_b_mean[:, None]
        
        Z_b = dX_b / np.sqrt(m - 1)
        Y_b = H_mat @ Z_b
        innovation = y_o - H_mat @ X_b_mean
        
        X_a = np.zeros((N, m))
        for i in range(N):
            R_loc_inv_diag = np.diag(R_loc_inv[i, :])
            P_a_tilde = np.linalg.inv((m -1) / delta *np.eye(m) + Y_b.T @ R_loc_inv_diag @ Y_b)
            
            w = P_a_tilde @ Y_b.T @ R_loc_inv_diag @ innovation
            
            X_a[i, :] = X_b_mean[i, 0] + dX_b[i, :] @ w
            
            D, C = np.linalg.eig(P_a_tilde)
            D = np.maximum(D, 1e-10)
            W = C @ np.diag(np.sqrt(D)) @ C.T
            
            Z_a = Z_b @ W